╔══════════════════════════════════════════════════════════════════════════╗
║   CATTLE AI — MASTER TRAINING NOTEBOOK  (ALL 5 MODELS)                ║
║   Kaggle GPU: T4 x2  |  Estimated time: 2–3 hours total               ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║  ┌─────────────────────────────────────────────────────────────────┐   ║
║  │  STEP 1 — ADD THESE 8 DATASETS IN KAGGLE BEFORE RUNNING        │   ║
║  │  Click "+ Add Data" (top right of notebook) → search each slug  │   ║
║  └─────────────────────────────────────────────────────────────────┘   ║
║                                                                        ║
║  FUNCTION 1 · Ear-tag Identification                                   ║
║  ├─ [PRIMARY] trainingdatapro/cows-detection-dataset                   ║
║  │   → YOLO bounding box labels, direct training input                 ║
║  └─ [VOLUME]  sadhliroomyprime/cattle-weight-detection-model-dataset-12k║
║      → 12k images, merged with primary for diversity                   ║
║                                                                        ║
║  FUNCTION 2 · Muzzle / Face Identification                             ║
║  ├─ [PRIMARY]  sharifashik/cow-muzzle-dataset                          ║
║  │   → Identity-labeled muzzle folders, ArcFace training               ║
║  ├─ [EXTRA-ID] kollabathulakaushik/cattle-images-db-for-muzzle-based-  ║
║  │              identification                                          ║
║  └─ [DIVERSITY] parthibanmarimuthu/beefcattle-muzzle                   ║
║      → Beef breed muzzles, cross-breed generalisation                  ║
║                                                                        ║
║  FUNCTION 3 · Body Condition Scoring                                   ║
║  ├─ [PRIMARY] alikhalilit98/cattle-body-parts-dataset-for-object-      ║
║  │             detection                                                ║
║  │   → Back/hip/rib labeled regions for BCS zone isolation             ║
║  └─ [TOP-VIEW] hienvuvg/mmcows                                         ║
║      → Multi-angle dairy cow images + metadata                         ║
║                                                                        ║
║  FUNCTION 4 · Lameness Detection                                       ║
║  ├─ [SHARED]  trainingdatapro/cows-detection-dataset  (same as Fn 1)   ║
║  │   → Side-view images for pose keypoint extraction                   ║
║  └─ [SENSOR]  sidqali/cow-monitoring-systemhourly-based-prediction     ║
║      → Hourly sensor time-series with locomotion labels for LSTM        ║
║                                                                        ║
║  FUNCTION 5 · Feeding Time / Behavior                                  ║
║  ├─ [PRIMARY] fandaoerji/cbvd-5cow-behavior-video-dataset              ║
║  │   → 5-class behavior labels: feeding/drinking/grooming/resting/     ║
║  │     walking                                                          ║
║  └─ [AUGMENT] lucyfirst/beef-cattle-behavior-data-set                  ║
║      → Beef breed behavior sequences, prevents dairy-only overfitting   ║
║                                                                        ║
║  STEP 2 — Set Kaggle accelerator: Settings → Accelerator → GPU T4 x2  ║
║  STEP 3 — Run all cells top to bottom                                  ║
║  STEP 4 — Download /kaggle/working/models/ after training completes    ║
╚══════════════════════════════════════════════════════════════════════════╝

─────────────────────────────────────────────────────────────────

In [50]:
pip install -q ultralytics timm albumentations onnx onnxruntime

Note: you may need to restart the kernel to use updated packages.


In [51]:
# CELL 1 ── Install dependencies
# ─────────────────────────────────────────────────────────────────
# !pip install -q ultralytics timm albumentations onnx onnxruntime

import os, gc, json, math, random, shutil, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
from torchvision import transforms
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, mean_absolute_error
from ultralytics import YOLO
import yaml

warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42); torch.manual_seed(42)

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = Path('/kaggle/working/models')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE} | Output: {OUT_DIR}')


# ─────────────────────────────────────────────────────────────────

Device: cuda | Output: /kaggle/working/models


In [52]:
# CELL 2 ── Verify all datasets are mounted
# ─────────────────────────────────────────────────────────────────

DATASETS = {
    'fn1_primary' : '/kaggle/input/datasets/raghavdharwal/cows-and-buffalo-computer-vision-dataset',
    'fn1_volume'  : '/kaggle/input/datasets/sadhliroomyprime/cattle-weight-detection-model-dataset-12k',
    'fn2_primary' : '/kaggle/input/datasets/sharifashik/cow-muzzle-dataset',
    'fn2_extra'   : '/kaggle/input/datasets/kollabathulakaushik/cattle-images-db-for-muzzle-based-identification',
    'fn2_beef'    : '/kaggle/input/datasets/parthibanmarimuthu/beefcattle-muzzle',
    'fn3_parts'   : '/kaggle/input/datasets/alikhalilit98/cattle-body-parts-dataset-for-object-detection',
    'fn3_mmcows'  : '/kaggle/input/datasets/hienvuvg/mmcows',
    'fn4_health'  : '/kaggle/input/datasets/khushupadhyay/cow-health-prediction',
    'fn4_symptoms'  : '/kaggle/input/datasets/researcher1548/livestock-symptoms-and-diseases',
    'fn5_cbvd'    : '/kaggle/input/datasets/fandaoerji/cbvd-5cow-behavior-video-dataset',
    'fn5_beef'    : '/kaggle/input/datasets/lucyfirst/beef-cattle-behavior-data-set',
}

print('\n── Dataset availability check ──')
# missing = []
# for key, path in DATASETS.items():
#     exists = Path(path).exists()
#     status = '✅' if exists else '❌ MISSING'
#     n_imgs = len(list(Path(path).rglob('*.jpg'))) if exists else 0
#     print(f'  {status}  {key:<15}  {path}  ({n_imgs} images)')
#     if not exists:
#         missing.append(key)

# if missing:
#     print(f'\n⚠️  {len(missing)} dataset(s) missing. Add them via "+ Add Data" in Kaggle.')
#     print('   Training will use synthetic fallback data for missing datasets.')
# else:
#     print('\n✅  All datasets mounted. Ready to train.')


# ═══════════════════════════════════════════════════════════════════════════
#  ███████╗ ██╗   ██╗ ███╗  ██╗  ██████╗ ████████╗ ██╗  ██████╗  ███╗  ██╗
#  ██╔════╝ ██║   ██║ ████╗ ██║ ██╔════╝    ██╔══╝ ██║ ██╔═══██╗ ████╗ ██║
#  █████╗   ██║   ██║ ██╔██╗██║ ██║         ██║    ██║ ██║   ██║ ██╔██╗██║
#  ██╔══╝   ██║   ██║ ██║╚████║ ██║         ██║    ██║ ██║   ██║ ██║╚████║
#  ██║      ╚██████╔╝ ██║ ╚███║ ╚██████╗    ██║    ██║ ╚██████╔╝ ██║ ╚███║
#  ╚═╝       ╚═════╝  ╚═╝  ╚══╝  ╚═════╝    ╚═╝    ╚═╝  ╚═════╝  ╚═╝  ╚══╝
#
#  FUNCTION 1 — EAR-TAG IDENTIFICATION
#  Datasets: cows-detection-dataset (primary) +
#            cattle-weight-detection-model-dataset-12k (volume)
#  Model:    YOLOv8n
#  Export:   TFLite int8 (eartag_detector.tflite)
# ═══════════════════════════════════════════════════════════════════════════

# print('\n' + '='*70)
# print('FUNCTION 1 ─ EAR-TAG IDENTIFICATION')
# print('  Datasets : cows-detection-dataset + cattle-weight-detection-model-dataset-12k')
# print('  Model    : YOLOv8n')
# print('='*70)

# WORK1 = Path('/kaggle/working/f1_eartag')
# for split in ['train', 'val']:
#     (WORK1 / 'images' / split).mkdir(parents=True, exist_ok=True)
#     (WORK1 / 'labels' / split).mkdir(parents=True, exist_ok=True)

# ── Collect images from BOTH datasets ──────────────────────────────────────
# Dataset 1: trainingdatapro/cows-detection-dataset
#   Structure: images/ + labels/ in YOLO format (most common)
# Dataset 2: sadhliroomyprime/cattle-weight-detection-model-dataset-12k
#   Structure: flat images, may or may not have labels

# D1_ROOT = Path(DATASETS['fn1_primary'])
# D2_ROOT = Path(DATASETS['fn1_volume'])

# all_imgs1 = []

# From dataset 1 — prefer images/ subfolder, else rglob
# img_dir1 = D1_ROOT / 'images' if (D1_ROOT / 'images').exists() else D1_ROOT
# all_imgs1 += list(img_dir1.rglob('*.jpg')) + list(img_dir1.rglob('*.png'))

# # From dataset 2 — large volume, use first 5000 to balance
# imgs2 = list(D2_ROOT.rglob('*.jpg')) + list(D2_ROOT.rglob('*.png'))
# random.shuffle(imgs2)
# all_imgs1 += imgs2[:5000]   # cap at 5000 from volume set

# random.shuffle(all_imgs1)
# n_val1 = max(1, int(len(all_imgs1) * 0.2))
# val_imgs1, train_imgs1 = all_imgs1[:n_val1], all_imgs1[n_val1:]

# def copy_with_label(img_path, split, data_root):
#     """Copy image and its YOLO label (or create pseudo-label) to working dir."""
#     dst_img = WORK1 / 'images' / split / img_path.name
#     shutil.copy(img_path, dst_img)

# Try to find existing YOLO label in multiple possible locations
# label_candidates = [
#         img_path.parent.parent / 'labels' / img_path.with_suffix('.txt').name,
#         img_path.with_suffix('.txt'),
#         img_path.parent / 'labels' / img_path.with_suffix('.txt').name,
#     ]
#     dst_lbl = WORK1 / 'labels' / split / img_path.with_suffix('.txt').name
#     lbl_found = next((p for p in label_candidates if p.exists()), None)

#     if lbl_found:
#         shutil.copy(lbl_found, dst_lbl)
#     else:
#         # Pseudo-label: full-image box for class 0 (cattle body)
#         # This teaches the detector WHERE cattle are; tag OCR is separate
#         dst_lbl.write_text('0 0.5 0.5 1.0 1.0\n')

# for p in train_imgs1: copy_with_label(p, 'train', D1_ROOT)
# for p in val_imgs1:   copy_with_label(p, 'val',   D1_ROOT)

# print(f'  Train images : {len(train_imgs1)}')
# print(f'  Val images   : {len(val_imgs1)}')
# print(f'  Source 1     : {len(list(D1_ROOT.rglob("*.jpg")))} images from cows-detection-dataset')
# print(f'  Source 2     : {min(5000, len(imgs2))} images from cattle-weight-dataset-12k')

# # Write YOLO data.yaml
# yaml1 = {
#     'path' : str(WORK1),
#     'train': 'images/train',
#     'val'  : 'images/val',
#     'nc'   : 1,
#     'names': ['ear_tag'],
# }
# yaml1_path = WORK1 / 'data.yaml'
# with open(yaml1_path, 'w') as f:
#     yaml.dump(yaml1, f)


── Dataset availability check ──


In [53]:
# ── CELL 3 ── Train YOLOv8n ────────────────────────────────────────────────
# yolov8n = nano (fastest, smallest, ~3MB export)
# 416px = optimal mobile size (balance of speed vs accuracy)
# model1 = YOLO('yolov8n.pt')
# results1 = model1.train(
#     data      = str(yaml1_path),
#     epochs    = 60,
#     imgsz     = 416,
#     batch     = 32,
#     lr0       = 0.01,
#     lrf       = 0.01,
#     momentum  = 0.937,
#     weight_decay = 0.0005,
#     warmup_epochs= 3,
#     project   = str(WORK1),
#     name      = 'run',
#     exist_ok  = True,
#     device    = 0 if DEVICE == 'cuda' else 'cpu',
#     patience  = 15,
#     save      = True,
#     plots     = True,
#     augment   = True,   # built-in mosaic + mixup augmentation
#     verbose   = False,
# )

In [54]:
# ── CELL 4 ── Evaluate ─────────────────────────────────────────────────────
# best1    = WORK1 / 'run' / 'weights' / 'best.pt'
# eval1    = YOLO(str(best1))
# metrics1 = eval1.val(data=str(yaml1_path), imgsz=416, verbose=False)
# print(f'\n[F1 RESULTS] mAP50: {metrics1.box.map50:.4f} | '
#       f'Precision: {metrics1.box.mp:.4f} | Recall: {metrics1.box.mr:.4f}')
# print(f'  Target: mAP50 > 0.70 on real ear-tag data')

In [55]:
# ── CELL 5 ── Export to TFLite (int8 quantised) ────────────────────────────
# eval1.export(format='tflite', imgsz=416, int8=True)
# tflite1 = list((WORK1 / 'run' / 'weights').glob('*int8.tflite'))
# if tflite1:
#     shutil.copy(tflite1[0], OUT_DIR / 'eartag_detector.tflite')
#     kb = tflite1[0].stat().st_size // 1024
#     print(f'✅ eartag_detector.tflite  ({kb} KB)')
# else:
#     # Fallback: save .pt weights
#     shutil.copy(best1, OUT_DIR / 'eartag_detector.pt')
#     print('✅ eartag_detector.pt  (TFLite export failed, using .pt)')

# json.dump({
#     'function'      : 'ear_tag_identification',
#     'model'         : 'YOLOv8n',
#     'datasets_used' : ['cows-detection-dataset', 'cattle-weight-detection-model-dataset-12k'],
#     'input_size'    : 416,
#     'input_dtype'   : 'float32',
#     'normalisation' : '0-1',
#     'output_format' : 'yolo_detection',
#     'classes'       : ['ear_tag'],
#     'conf_threshold': 0.4,
#     'iou_threshold' : 0.5,
#     'mAP50'         : round(metrics1.box.map50, 4),
#     'flutter_model' : 'assets/models/eartag_detector.tflite',
#     'inference_note': 'After detecting tag region, crop and run TrOCR or digit-CNN for number reading',
# }, open(OUT_DIR / 'eartag_meta.json', 'w'), indent=2)

# del model1, eval1; gc.collect(); torch.cuda.empty_cache()


# ═══════════════════════════════════════════════════════════════════════════
#  FUNCTION 2 — MUZZLE / FACE IDENTIFICATION
#  Datasets: cow-muzzle-dataset (primary, identity folders)
#            cattle-images-db-for-muzzle-based-identification (extra IDs)
#            beefcattle-muzzle (beef breed diversity)
# #  Model:    ResNet-50 backbone + ArcFace metric learning
# #  Export:   .pth weights (muzzle_embedder.pth)
# # ═══════════════════════════════════════════════════════════════════════════

# print('\n' + '='*70)
# print('FUNCTION 2 ─ MUZZLE / FACE IDENTIFICATION')
# print('  Datasets : cow-muzzle-dataset (primary)')
# print('           + cattle-images-db-for-muzzle-based-identification (extra IDs)')
# print('           + beefcattle-muzzle (breed diversity)')
# print('  Model    : ResNet-50 + ArcFace loss')
# print('='*70)

# WORK2 = Path('/kaggle/working/f2_muzzle')
# WORK2.mkdir(parents=True, exist_ok=True)

In [56]:
# ── CELL 6 ── Collect muzzle images from all 3 datasets ────────────────────
# All 3 datasets must have folder-per-identity structure:
#   dataset_root/
#     cow_001/  img1.jpg  img2.jpg ...   ← each folder = one cow identity
#     cow_002/  img1.jpg ...

# id_to_imgs2 = defaultdict(list)

# def load_identity_folder(root_path, prefix=''):
#     """Load folder-per-identity structure. prefix avoids ID collisions between datasets."""
#     root = Path(root_path)
#     if not root.exists():
#         return
#     for d in root.iterdir():
#         if d.is_dir():
#             imgs = list(d.glob('*.jpg')) + list(d.glob('*.png'))
#             if len(imgs) >= 2:
#                 uid = f'{prefix}_{d.name}'   # e.g. 'dairy_cow_001', 'beef_cow_042'
#                 id_to_imgs2[uid].extend(imgs)

# load_identity_folder(DATASETS['fn2_primary'], prefix='dairy')
# load_identity_folder(DATASETS['fn2_extra'],   prefix='db')
# load_identity_folder(DATASETS['fn2_beef'],    prefix='beef')

# # If no identity folders found, try flat structure with synthetic IDs
# if not id_to_imgs2:
#     print('  ⚠️  No identity folders found. Building synthetic identity groups.')
#     all_flat = (list(Path(DATASETS['fn2_primary']).rglob('*.jpg')) +
#                 list(Path(DATASETS['fn2_extra']).rglob('*.jpg'))   +
#                 list(Path(DATASETS['fn2_beef']).rglob('*.jpg')))
# random.shuffle(all_flat)
    # Group every 5 images as one identity (pseudo-labelling)
#     for i in range(0, len(all_flat)-5, 5):
#         uid = f'cow_{i//5:04d}'
#         id_to_imgs2[uid] = all_flat[i:i+5]

# cow_ids2   = sorted(id_to_imgs2.keys())
# n_classes2 = len(cow_ids2)
# id2lbl2    = {cid: i for i, cid in enumerate(cow_ids2)}

# total_imgs2 = sum(len(v) for v in id_to_imgs2.values())
# print(f'  Identities (classes) : {n_classes2}')
# print(f'  Total images         : {total_imgs2}')
# print(f'  Avg images per cow   : {total_imgs2 / max(n_classes2,1):.1f}')

# train_items2, val_items2 = [], []
# for cid, imgs in id_to_imgs2.items():
#     random.shuffle(imgs)
#     n_v = max(1, int(len(imgs) * 0.2))
#     val_items2   += [(p, id2lbl2[cid]) for p in imgs[:n_v]]
#     train_items2 += [(p, id2lbl2[cid]) for p in imgs[n_v:]]

# IMG_SIZE2 = 224
# tf_train2 = transforms.Compose([
#     transforms.Resize((IMG_SIZE2, IMG_SIZE2)),
#     transforms.RandomHorizontalFlip(),
#     transforms.ColorJitter(0.3, 0.3, 0.2),
#     transforms.RandomRotation(15),
#     transforms.RandomGrayscale(p=0.05),   # muzzle texture invariant to colour
#     transforms.ToTensor(),
#     transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
# ])
# tf_val2 = transforms.Compose([
#     transforms.Resize((IMG_SIZE2, IMG_SIZE2)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
# ])

# class MuzzleDS(Dataset):
#     def __init__(self, items, tf): self.items=items; self.tf=tf
#     def __len__(self): return len(self.items)
#     def __getitem__(self, i):
#         p, lbl = self.items[i]
#         try:    img = Image.open(p).convert('RGB')
#         except: img = Image.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8))
#         return self.tf(img), torch.tensor(lbl, dtype=torch.long)

# dl_tr2  = DataLoader(MuzzleDS(train_items2, tf_train2), 32, shuffle=True,  num_workers=2, pin_memory=True)
# dl_val2 = DataLoader(MuzzleDS(val_items2,   tf_val2),   32, shuffle=False, num_workers=2)
# print(f'  Train: {len(train_items2)} | Val: {len(val_items2)}')

In [57]:
# # ── CELL 7 ── ArcFace loss ─────────────────────────────────────────────────
# class ArcFaceLoss(nn.Module):
#     """
#     ArcFace: Additive Angular Margin Loss.
#     Forces same-identity embeddings to cluster tightly in hypersphere.
#     s=64 (scale), m=0.5 (margin in radians ≈ 28.6°)
#     """
#     def __init__(self, dim, n_cls, s=64.0, m=0.50):
#         super().__init__()
#         self.s=s; self.m=m
    #     self.W = nn.Parameter(torch.FloatTensor(n_cls, dim))
    #     nn.init.xavier_uniform_(self.W)
    #     self.ce=nn.CrossEntropyLoss()
    #     self.cos_m=math.cos(m); self.sin_m=math.sin(m)
    #     self.th=math.cos(math.pi-m); self.mm=math.sin(math.pi-m)*m

    # def forward(self, emb, lbl):
    #     e=F.normalize(emb,dim=1); w=F.normalize(self.W,dim=1)
    #     cos=F.linear(e,w); sin=(1-cos.pow(2).clamp(0,1)).sqrt()
    #     phi=cos*self.cos_m - sin*self.sin_m
    #     phi=torch.where(cos>self.th, phi, cos-self.mm)
    #     oh=torch.zeros_like(cos).scatter_(1,lbl.view(-1,1),1.0)
    #     return self.ce((oh*phi+(1-oh)*cos)*self.s, lbl)

In [58]:
# # ── CELL 8 ── ResNet-50 embedder ───────────────────────────────────────────
# EMBED_DIM2 = 256  # 256-d: fast cosine search on Flutter device

# class MuzzleEmbedder(nn.Module):
#     def __init__(self):
#         super().__init__()
#         bb=tvm.resnet50(weights='IMAGENET1K_V2'); bb.fc=nn.Identity()
#         self.bb=bb
#         self.head=nn.Sequential(
#             nn.Linear(2048,512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.4),
#             nn.Linear(512, EMBED_DIM2), nn.BatchNorm1d(EMBED_DIM2),
#         )
#     def forward(self, x): return self.head(self.bb(x))

# model2   = MuzzleEmbedder().to(DEVICE)
# arcface2 = ArcFaceLoss(EMBED_DIM2, n_classes2).to(DEVICE)
# opt2     = torch.optim.AdamW(
#     list(model2.parameters())+list(arcface2.parameters()), lr=3e-4, weight_decay=1e-4)
# sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=40, eta_min=1e-6)

In [59]:
# # ── CELL 9 ── Train ArcFace ────────────────────────────────────────────────
# # best_acc2 = 0.0
# # for epoch in range(1, 41):
# #     model2.train(); arcface2.train()
# #     total_loss = 0
# #     for imgs, lbls in dl_tr2:
# #         imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
# #         opt2.zero_grad()
# #         loss = arcface2(model2(imgs), lbls)
# #         loss.backward()
# #         nn.utils.clip_grad_norm_(model2.parameters(), 5.0)
# #         opt2.step()
# #         total_loss += loss.item()
# #     sch2.step()

# #     if epoch % 5 == 0 or epoch == 40:
# #         model2.eval()
#         # embs, labs = [], []
#         # with torch.no_grad():
#         #     for imgs, lbls in dl_val2:
#         #         embs.append(F.normalize(model2(imgs.to(DEVICE)),dim=1).cpu()); labs.append(lbls)
#         # embs=torch.cat(embs); labs=torch.cat(labs)
#         # gall_e, gall_l = [], []
#         # with torch.no_grad():
#         #     for imgs, lbls in dl_tr2:
#         #         gall_e.append(F.normalize(model2(imgs.to(DEVICE)),dim=1).cpu()); gall_l.append(lbls)
#         # gall_e=torch.cat(gall_e); gall_l=torch.cat(gall_l)
#         # preds=(embs @ gall_e.T).argmax(dim=1)
#         # acc=float((gall_l[preds]==labs).float().mean())
#         # print(f'  Epoch {epoch:03d} | Loss: {total_loss/len(dl_tr2):.4f} | Rank-1: {acc:.4f}')
#         # if acc > best_acc2:
#         #     best_acc2=acc
#         #     torch.save(model2.state_dict(), WORK2/'best_muzzle.pth')

# # print(f'  Best Rank-1: {best_acc2:.4f}')

# # shutil.copy(WORK2/'best_muzzle.pth', OUT_DIR/'muzzle_embedder.pth')
# # json.dump({
# #     'function'      : 'muzzle_identification',
# #     'model'         : 'ResNet50 + ArcFace',
# #     'datasets_used' : ['cow-muzzle-dataset', 'cattle-images-db-for-muzzle-based-identification', 'beefcattle-muzzle'],
# #     'input_size'    : IMG_SIZE2, 'embed_dim': EMBED_DIM2,
# #     'input_dtype'   : 'float32',
# #     'normalize_mean': [0.485,0.456,0.406], 'normalize_std': [0.229,0.224,0.225],
# #     'similarity'    : 'cosine',
# #     'rank1_accuracy': round(best_acc2,4),
# #     'flutter_model' : 'assets/models/muzzle_embedder.pth',
# #     'inference_note': 'L2-normalise output embedding before cosine similarity with gallery',
# #     'classes_trained': n_classes2,
# # }, open(OUT_DIR/'muzzle_meta.json','w'), indent=2)
# # print('✅ muzzle_embedder.pth saved')
# # del model2, arcface2; gc.collect(); torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════════
# #  FUNCTION 3 — BODY CONDITION SCORING (BCS)
# #  Datasets: cattle-body-parts-dataset-for-object-detection (primary)
# #            mmcows (top-view multi-angle)
# #  Model:    EfficientNet-B3 + ordinal regression head
# #  Export:   .pth weights (bcs_scorer.pth)
# # ═══════════════════════════════════════════════════════════════════════════

# print('\n' + '='*70)
# print('FUNCTION 3 ─ BODY CONDITION SCORING')
# print('  Datasets : cattle-body-parts-dataset-for-object-detection (primary)')
# print('           + mmcows (top-view, multi-angle)')
# print('  Model    : EfficientNet-B3 + ordinal regression')
# print('='*70)

# WORK3 = Path('/kaggle/working/f3_bcs')
# WORK3.mkdir(parents=True, exist_ok=True)

# D3a_ROOT = Path(DATASETS['fn3_parts'])   # body parts dataset
# D3b_ROOT = Path(DATASETS['fn3_mmcows'])  # mmcows dataset

In [60]:
# # ── CELL 10 ── Load images + BCS labels ────────────────────────────────────
# # Strategy:
# #   1. Try to load existing CSV with BCS labels from either dataset
# #   2. If no CSV, use folder names as BCS proxy (e.g. folders named 'bcs_3' or '3.0')
# #   3. Final fallback: synthetic labels for pipeline demonstration

# def load_bcs_csv(root):
#     """Search for CSV with BCS labels in dataset root."""
    
#     csvs = list(Path(root).rglob('*.csv'))

#     for csv in csvs:
#         df = pd.read_csv(csv)
#         df.columns = [c.lower().strip() for c in df.columns]

#         # Look for BCS column under various names
#         for col in ['bcs', 'score', 'body_condition', 'condition_score', 'bcs_score']:
#             if col in df.columns:
#                 df = df.rename(columns={col: 'bcs'})
#                 print(f'  Found BCS labels in {csv.name}: {len(df)} rows')
#                 return df

#     return None

# df3a = load_bcs_csv(DATASETS['fn3_parts'])
# df3b = load_bcs_csv(DATASETS['fn3_mmcows'])

# # Collect all images from both datasets
# all_imgs3a = list(D3a_ROOT.rglob('*.jpg')) + list(D3a_ROOT.rglob('*.png'))
# all_imgs3b = list(D3b_ROOT.rglob('*.jpg')) + list(D3b_ROOT.rglob('*.png'))
# all_imgs3  = all_imgs3a + all_imgs3b

# if df3a is not None or df3b is not None:
#     # Merge available CSVs
#     frames = [d for d in [df3a, df3b] if d is not None]
#     df3 = pd.concat(frames, ignore_index=True)

#     # Resolve image paths
#     def resolve(p):
#         if p and Path(str(p)).exists():
#             return str(p)

#         for root in [D3a_ROOT, D3b_ROOT]:
#             c = root / str(p)
#             if c.exists():
#                 return str(c)

#         return None

#     if 'image_path' not in df3.columns:
#         for col in ['image', 'filename', 'file', 'name']:
#             if col in df3.columns:
#                 df3 = df3.rename(columns={col: 'image_path'})
#                 break

#     df3['image_path'] = df3.get(
#         'image_path',
#         pd.Series([str(p) for p in all_imgs3[:len(df3)]])
#     ).apply(resolve)

#     df3 = df3.dropna(subset=['image_path'])

# else:
#     print(f'  No BCS CSV found. Generating synthetic labels for {len(all_imgs3)} images.')
#     print('  Replace synthetic labels with expert BCS annotations for real accuracy.')

#     df3 = pd.DataFrame({
#         'image_path': [str(p) for p in all_imgs3],
#         'bcs': np.random.uniform(1.5, 4.5, len(all_imgs3)).round(1),
#     })

# df3['bcs'] = pd.to_numeric(df3['bcs'], errors='coerce').clip(1.0, 5.0).fillna(3.0)

# print(f'  Total labeled samples: {len(df3)}')
# print(f'  BCS distribution:\n{df3["bcs"].describe().round(2)}')
# # ── LIMIT DATASET TO ONLY 10,000 SAMPLES ─────────────────────

# MAX_SAMPLES3 = 10000

# if len(df3) > MAX_SAMPLES3:

#     df3 = df3.sample(
#         n=MAX_SAMPLES3,
#         random_state=42
#     ).reset_index(drop=True)

#     print(f'✅ Using only {MAX_SAMPLES3} labeled samples')

# else:
#     print(f'✅ Dataset already smaller than {MAX_SAMPLES3}')

# train_df3, val_df3 = train_test_split(df3, test_size=0.2, random_state=42)

# # 8 ordinal thresholds covering BCS 1.0 → 5.0
# BCS_BINS3 = np.linspace(1.0, 5.0, 9).tolist()

# def bcs_to_ordinal(bcs):
#     """BCS float → 8-bit binary ordinal vector."""
#     return torch.tensor(
#         [1.0 if bcs > BCS_BINS3[k] else 0.0 for k in range(8)],
#         dtype=torch.float32
#     )

# def ordinal_to_bcs(logits):
#     """8-d logits → BCS float via threshold counting."""
#     probs = torch.sigmoid(logits)
#     k = min(int((probs > 0.5).sum().item()), 8)
#     return float(BCS_BINS3[k])

# aug3_train = A.Compose([
#     A.Resize(384, 384),
#     A.HorizontalFlip(p=0.5),
#     A.VerticalFlip(p=0.1),
#     A.RandomBrightnessContrast(0.25, 0.25, p=0.5),
#     A.HueSaturationValue(10, 20, 10, p=0.3),
#     A.ShiftScaleRotate(0.05, 0.1, 15, p=0.4),
#     A.CoarseDropout(max_holes=6, max_height=32, max_width=32, p=0.25),
#     A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
#     ToTensorV2(),
# ])

# aug3_val = A.Compose([
#     A.Resize(384,384),
#     A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
#     ToTensorV2(),
# ])

# class BCSDS(Dataset):
#     def __init__(self, df, aug):
#         self.df = df.reset_index(drop=True)
#         self.aug = aug

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, i):
#         row = self.df.iloc[i]

#         try:
#             img = cv2.cvtColor(
#                 cv2.imread(str(row['image_path'])),
#                 cv2.COLOR_BGR2RGB
#             )
#         except:
#             img = np.random.randint(0,255,(384,384,3),dtype=np.uint8)

#         x = self.aug(image=img)['image']
#         bcs = float(row['bcs'])

#         return x, bcs_to_ordinal(bcs), torch.tensor(bcs)

# dl_tr3 = DataLoader(
#     BCSDS(train_df3, aug3_train),
#     16,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True
# )

# dl_val3 = DataLoader(
#     BCSDS(val_df3, aug3_val),
#     16,
#     shuffle=False,
#     num_workers=2
# )

In [61]:
# # ── CELL 11 ── EfficientNet-B3 model ──────────────────────────────────────
# class BCSModel(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.bb=timm.create_model('efficientnet_b3', pretrained=True, num_classes=0, global_pool='avg')
#         self.head=nn.Sequential(
#             nn.Dropout(0.35), nn.Linear(1536,512), nn.SiLU(),
#             nn.Dropout(0.2),  nn.Linear(512, 8),
#         )
#     def forward(self, x): return self.head(self.bb(x))

# model3  = BCSModel().to(DEVICE)
# crit3   = nn.BCEWithLogitsLoss()
# opt3    = torch.optim.AdamW(model3.parameters(), lr=2e-4, weight_decay=1e-4)
# sch3    = torch.optim.lr_scheduler.OneCycleLR(opt3, max_lr=2e-4,
#            steps_per_epoch=len(dl_tr3), epochs=50)
# # ── CHECKPOINT RESUME SUPPORT ────────────────────────────────

# checkpoint_path = WORK3 / 'bcs_checkpoint.pth'

# best_mae3 = float('inf')
# start_epoch = 1

# if checkpoint_path.exists():

#     checkpoint = torch.load(checkpoint_path)

#     model3.load_state_dict(checkpoint['model_state_dict'])
#     opt3.load_state_dict(checkpoint['optimizer_state_dict'])
#     sch3.load_state_dict(checkpoint['scheduler_state_dict'])

#     best_mae3 = checkpoint['best_mae']
#     start_epoch = checkpoint['epoch'] + 1

#     print(f'✅ Resuming from epoch {start_epoch}')

# else:
#     print('🚀 Starting fresh training')

# for epoch in range(start_epoch, 51):
#     model3.train()
#     for x, ord_lbl, _ in dl_tr3:
#         x, ord_lbl = x.to(DEVICE), ord_lbl.to(DEVICE)
#         opt3.zero_grad()
#         crit3(model3(x), ord_lbl).backward()
#         nn.utils.clip_grad_norm_(model3.parameters(), 2.0)
#         opt3.step(); sch3.step()

#     if epoch % 5 == 0 or epoch == 50:
#         model3.eval()
#         preds3, trues3 = [], []
#         with torch.no_grad():
#             for x, _, bcs_true in dl_val3:
#                 lg=model3(x.to(DEVICE)).cpu()
#                 preds3 += [ordinal_to_bcs(lg[i]) for i in range(lg.size(0))]
#                 trues3 += bcs_true.tolist()
#         mae3 = mean_absolute_error(trues3, preds3)
#         print(f'  Epoch {epoch:03d} | MAE: {mae3:.3f}  (target < 0.25)')
#         if mae3 < best_mae3:
#             best_mae3=mae3
#             torch.save(model3.state_dict(), WORK3/'best_bcs.pth')
#             # ── SAVE CHECKPOINT EVERY EPOCH ─────────────────────────────

#             torch.save({
#               'epoch': epoch,
#               'model_state_dict': model3.state_dict(),
#               'optimizer_state_dict': opt3.state_dict(),
#               'scheduler_state_dict': sch3.state_dict(),
#               'best_mae': best_mae3,
#               }, checkpoint_path)

#               print(f'💾 Checkpoint saved for epoch {epoch}')

# print(f'  Best MAE: {best_mae3:.3f}')
# shutil.copy(WORK3/'best_bcs.pth', OUT_DIR/'bcs_scorer.pth')
# json.dump({
#     'function'      : 'body_condition_scoring',
#     'model'         : 'EfficientNet-B3 + ordinal regression',
#     'datasets_used' : ['cattle-body-parts-dataset-for-object-detection', 'mmcows'],
#     'input_size'    : 384, 'input_dtype': 'float32',
#     'output_size'   : 8,   'output_type': 'ordinal_logits',
#     'bcs_bins'      : BCS_BINS3,
#     'normalize_mean': [0.485,0.456,0.406], 'normalize_std': [0.229,0.224,0.225],
#     'best_mae'      : round(best_mae3,4),
#     'flutter_model' : 'assets/models/bcs_scorer.pth',
#     'inference_note': 'sigmoid(output) → count entries >0.5 → index bcs_bins',
# }, open(OUT_DIR/'bcs_meta.json','w'), indent=2)
# print('✅ bcs_scorer.pth saved')
# del model3; gc.collect(); torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════════
# #  FUNCTION 4 — LAMENESS DETECTION
# #  Datasets: cows-detection-dataset (pose keypoint images, shared with Fn 1)
# #            cow-monitoring-systemhourly-based-prediction (sensor LSTM data)
# #            cattle-body-parts-dataset (spine/hip keypoints, shared with Fn 3)
# #  Model:    YOLOv8n-pose (keypoint extraction) + BiLSTM (temporal gait)
# #  Export:   .pth weights (lameness_detector.pth)
# # ═══════════════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('FUNCTION 4 ─ LAMENESS DETECTION')
print('  Datasets : cows-detection-dataset (pose images, shared Fn 1)')
print('           + cow-monitoring-systemhourly-based-prediction (LSTM sensor data)')
print('           + cattle-body-parts-dataset (spine/hip, shared Fn 3)')
print('  Model    : YOLOv8n-pose → BiLSTM classifier')
print('='*70)

WORK4 = Path('/kaggle/working/f4_lameness')
WORK4.mkdir(parents=True, exist_ok=True)

In [62]:
# ── CELL 12 ── Load sensor dataset (primary LSTM training source) ──────────
SENSOR_ROOT = Path(DATASETS['fn4_health', 'fn4_symptoms'])
SEQ_LEN4    = 20     # 20 time steps per gait sequence
FEAT_DIM4   = 17 * 3 # YOLOv8-pose: 17 keypointJJ s × (x, y, confidence)

sensor_seqs, sensor_labs = [], []

# Try to load hourly sensor CSV
sensor_csvs = list(SENSOR_ROOT.rglob('*.csv'))
if sensor_csvs:
    print(f'  Loading sensor data from {sensor_csvs[0].name}')
    df_sensor = pd.read_csv(sensor_csvs[0])
    df_sensor.columns = [c.lower().strip() for c in df_sensor.columns]
    print(f'  Columns: {list(df_sensor.columns)}')

    # Common column names in locomotion datasets
    label_col = next((c for c in df_sensor.columns
                      if any(k in c for k in ['lame','label','score','class','locomotion'])), None)
    feature_cols = [c for c in df_sensor.columns
                    if c not in [label_col,'cow_id','date','time','id'] and label_col]

    if label_col and len(feature_cols) >= 3:
        # Build sequences from sensor rows (sliding window)
        feat_arr = df_sensor[feature_cols].fillna(0).values.astype(np.float32)
        lbl_arr  = (df_sensor[label_col].fillna(0) > 1).astype(int).values  # >1 = lame

        # Pad features to FEAT_DIM4
        if feat_arr.shape[1] < FEAT_DIM4:
            pad = np.zeros((len(feat_arr), FEAT_DIM4-feat_arr.shape[1]), dtype=np.float32)
            feat_arr = np.hstack([feat_arr, pad])
        else:
            feat_arr = feat_arr[:, :FEAT_DIM4]

        for start in range(0, len(feat_arr)-SEQ_LEN4+1, 5):
            seq = feat_arr[start:start+SEQ_LEN4]
            lbl = int(lbl_arr[start+SEQ_LEN4//2])  # label at midpoint of window
            sensor_seqs.append(seq); sensor_labs.append(lbl)
        print(f'  Sensor sequences built: {len(sensor_seqs)} (lame: {sum(sensor_labs)})')

In [63]:
# ── CELL 13 ── Extract pose keypoints from images ─────────────────────────
pose_model4 = YOLO('yolov8n-pose.pt')
IMG_ROOT4   = Path(DATASETS['fn1_primary'])    # side-view images
img_seqs4, img_labs4 = [], []

class_dirs4 = [d for d in IMG_ROOT4.iterdir() if d.is_dir()]
for d in class_dirs4:
    imgs = sorted(d.glob('*.jpg'))[:100]  # cap per class for speed
    lbl  = 1 if 'lame' in d.name.lower() else 0
    seq_buf = []
    for img_path in imgs:
        res = pose_model4.predict(str(img_path), verbose=False)
        if res and res[0].keypoints is not None and res[0].keypoints.data.shape[0] > 0:
            kp = res[0].keypoints.data[0].reshape(-1).cpu().numpy().astype(np.float32)
        else:
            kp = np.zeros(FEAT_DIM4, dtype=np.float32)
        seq_buf.append(kp)
        if len(seq_buf) >= SEQ_LEN4:
            img_seqs4.append(np.array(seq_buf[-SEQ_LEN4:]))
            img_labs4.append(lbl)

print(f'  Pose-based sequences: {len(img_seqs4)}')

# Merge sensor + pose sequences
all_seqs4 = sensor_seqs + img_seqs4
all_labs4  = sensor_labs + img_labs4

if not all_seqs4:
    print('  ⚠️  No real sequences. Generating synthetic gait data.')
    for _ in range(400):
        lbl = random.randint(0, 1)
        seq = np.random.randn(SEQ_LEN4, FEAT_DIM4).astype(np.float32)
        if lbl == 1:
            seq[:, 39:42] += np.random.randn(SEQ_LEN4, 3) * 0.6
        all_seqs4.append(seq); all_labs4.append(lbl)

seqs4  = np.array(all_seqs4, dtype=np.float32)
labs4  = np.array(all_labs4, dtype=np.int64)
idx    = np.random.permutation(len(seqs4))
n_v4   = max(1, int(len(idx)*0.2))
v_idx, t_idx = idx[:n_v4], idx[n_v4:]

class SeqDS(Dataset):
    def __init__(self, s, l): self.s=s; self.l=l
    def __len__(self): return len(self.s)
    def __getitem__(self, i): return torch.tensor(self.s[i]), torch.tensor(self.l[i])

dl_tr4  = DataLoader(SeqDS(seqs4[t_idx], labs4[t_idx]), 32, shuffle=True)
dl_val4 = DataLoader(SeqDS(seqs4[v_idx], labs4[v_idx]), 32)
print(f'  Train seqs: {len(t_idx)} | Val seqs: {len(v_idx)} | '
      f'Lame rate: {labs4.mean():.1%}')

class LamenessLSTM(nn.Module):
    """
    Bidirectional LSTM with temporal attention.
    Processes a 20-frame keypoint sequence and classifies healthy vs lame.
    """
    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(FEAT_DIM4)
        self.proj = nn.Linear(FEAT_DIM4, 128)
        self.lstm = nn.LSTM(128, 128, num_layers=2, batch_first=True,
                            dropout=0.3, bidirectional=True)
        self.attn = nn.Linear(256, 1)
        self.cls  = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(256,64), nn.ReLU(), nn.Linear(64,2))
    def forward(self, x):
        x=self.proj(F.relu(self.norm(x)))
        out,_=self.lstm(x)
        w=torch.softmax(self.attn(out),dim=1)
        ctx=(w*out).sum(dim=1)
        return self.cls(ctx)

model4 = LamenessLSTM().to(DEVICE)
crit4  = nn.CrossEntropyLoss(weight=torch.tensor([1.0,2.5]).to(DEVICE))
opt4   = torch.optim.AdamW(model4.parameters(), lr=5e-4, weight_decay=1e-4)
sch4   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt4,'max',patience=5,factor=0.5)
best_f1_4 = 0.0

for epoch in range(1, 41):
    model4.train()
    for seqs, labs in dl_tr4:
        seqs,labs=seqs.to(DEVICE),labs.to(DEVICE)
        opt4.zero_grad(); crit4(model4(seqs),labs).backward()
        nn.utils.clip_grad_norm_(model4.parameters(),1.0); opt4.step()
    model4.eval()
    p4,l4=[],[]
    with torch.no_grad():
        for seqs,labs in dl_val4:
            p4+=model4(seqs.to(DEVICE)).argmax(1).cpu().tolist(); l4+=labs.tolist()
    f14=f1_score(l4,p4,average='weighted',zero_division=0); sch4.step(f14)
    if epoch%5==0: print(f'  Epoch {epoch:03d} | F1: {f14:.4f}')
    if f14>best_f1_4: best_f1_4=f14; torch.save(model4.state_dict(), WORK4/'best_lameness.pth')

print(f'  Best F1: {best_f1_4:.4f}')
shutil.copy(WORK4/'best_lameness.pth', OUT_DIR/'lameness_detector.pth')
json.dump({
    'function'      : 'lameness_detection',
    'model'         : 'YOLOv8n-pose + BiLSTM (bidirectional with attention)',
    'datasets_used' : ['cows-detection-dataset', 'cow-monitoring-systemhourly-based-prediction',
                       'cattle-body-parts-dataset-for-object-detection'],
    'seq_len'       : SEQ_LEN4, 'feat_dim': FEAT_DIM4, 'n_keypoints': 17,
    'input_dtype'   : 'float32',
    'classes'       : ['healthy','lame'],
    'best_f1'       : round(best_f1_4,4),
    'flutter_model' : 'assets/models/lameness_detector.pth',
    'inference_note': 'Extract 20-frame keypoint sequence via YOLOv8-pose, stack as [20,51] tensor',
}, open(OUT_DIR/'lameness_meta.json','w'), indent=2)
print('✅ lameness_detector.pth saved')
del model4, pose_model4; gc.collect(); torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════════
# #  FUNCTION 5 — FEEDING TIME / BEHAVIOR CLASSIFICATION
# #  Datasets: cbvd-5cow-behavior-video-dataset (primary, 5 labeled classes)
# #            beef-cattle-behavior-data-set (augment, cross-breed)
# #  Model:    MobileNetV3-Small (lightweight, fast on Flutter device)
# #  Export:   .pth weights (behavior_classifier.pth)
# # ═══════════════════════════════════════════════════════════════════════════

# print('\n' + '='*70)
# print('FUNCTION 5 ─ FEEDING TIME / BEHAVIOR CLASSIFICATION')
# print('  Datasets : cbvd-5cow-behavior-video-dataset (primary, 5 classes)')
# print('           + beef-cattle-behavior-data-set (augment, cross-breed)')
# print('  Model    : MobileNetV3-Small')
# print('='*70)

# WORK5 = Path('/kaggle/working/f5_behavior')
# WORK5.mkdir(parents=True, exist_ok=True)

# D5a_ROOT = Path(DATASETS['fn5_cbvd'])   # CBVD-5: feeding/drinking/grooming/resting/walking
# D5b_ROOT = Path(DATASETS['fn5_beef'])   # Beef cattle behaviors

# CLASSES5 = ['feeding','drinking','standing','resting','walking']
# cls2id5  = {c:i for i,c in enumerate(CLASSES5)}


FUNCTION 5 ─ FEEDING TIME / BEHAVIOR CLASSIFICATION
  Datasets : cbvd-5cow-behavior-video-dataset (primary, 5 classes)
           + beef-cattle-behavior-data-set (augment, cross-breed)
  Model    : MobileNetV3-Small


In [64]:
# # ── CELL 14 ── Load images from both datasets ──────────────────────────────
# items5 = []

# # Add flexible aliases for behavior names
# CLASS_ALIASES5 = {
#     'feeding' : ['feeding', 'feed', 'eating', 'eat', 'grazing'],
#     'resting' : ['resting', 'rest', 'lying', 'lie', 'idle', 'sleeping'],
#     'walking' : ['walking', 'walk', 'moving'],
#     'standing': ['standing', 'stand'],
#     'drinking': ['drinking', 'drink']
# }

# def load_behavior_dataset(root, prefix=''):
#     """
#     Recursively load behavior images from any folder structure.
#     """

#     loaded = 0
#     root = Path(root)

#     if not root.exists():
#         print(f'  Missing dataset: {root}')
#         return loaded

#     # Search images recursively but LIMIT total images
#     MAX_IMAGES_PER_DATASET = 10000

#     imgs = (
#         list(root.rglob('*.jpg')) +
#         list(root.rglob('*.png')) +
#         list(root.rglob('*.jpeg'))
#     )

#     random.shuffle(imgs)

#     imgs = imgs[:MAX_IMAGES_PER_DATASET]

#     print(f'  Using {len(imgs)} images from {prefix}')

#     for p in imgs:

#         path_lower = str(p).lower()

#         for cls_name, aliases in CLASS_ALIASES5.items():

#             matched = False

#             for alias in aliases:

#                 if alias in path_lower:

#                     if cls_name not in cls2id5:
#                         cls2id5[cls_name] = len(cls2id5)

#                     items5.append((str(p), cls2id5[cls_name]))
#                     loaded += 1
#                     matched = True
#                     break

#             if matched:
#                 break

#     return loaded
         

# n5a = load_behavior_dataset(DATASETS['fn5_cbvd'], 'CBVD-5')
# n5b = load_behavior_dataset(DATASETS['fn5_beef'], 'BeefBehavior')

# print(f'  CBVD-5 images    : {n5a}')
# print(f'  Beef behavior    : {n5b}')
# print(f'  Total items      : {len(items5)}')


# # If still empty -> synthetic fallback
# if not items5:
#     print('  ⚠️  No labeled folders found. Generating synthetic data.')

#     for _ in range(1000):
#         items5.append((None, random.randint(0, 4)))


# # Class distribution
# from collections import Counter

# cnt5 = Counter(lbl for _, lbl in items5)

# for cname, idx in cls2id5.items():
#     print(f'  {cname:<12}: {cnt5.get(idx,0):>5} images')


# random.shuffle(items5)

# n_v5 = int(len(items5) * 0.2)

# val5 = items5[:n_v5]
# tr5  = items5[n_v5:]


# tf5_train = transforms.Compose([
#     transforms.Resize((224,224)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(10),
#     transforms.ColorJitter(0.3,0.3,0.2,0.1),
#     transforms.RandomGrayscale(p=0.05),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         [0.485,0.456,0.406],
#         [0.229,0.224,0.225]
#     ),
# ])

# tf5_val = transforms.Compose([
#     transforms.Resize((224,224)),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         [0.485,0.456,0.406],
#         [0.229,0.224,0.225]
#     ),
# ])


# class BehaviorDS(Dataset):

#     def __init__(self, items, tf):
#         self.items = items
#         self.tf    = tf

#     def __len__(self):
#         return len(self.items)

#     def __getitem__(self, i):

#         p, lbl = self.items[i]

#         try:
#             if p and Path(p).exists():
#                 img = Image.open(p).convert('RGB')
#             else:
#                 img = Image.fromarray(
#                     np.random.randint(
#                         0,255,(224,224,3),
#                         dtype=np.uint8
#                     )
#                 )

#         except:
#             img = Image.fromarray(
#                 np.random.randint(
#                     0,255,(224,224,3),
#                     dtype=np.uint8
#                 )
#             )

#         return self.tf(img), torch.tensor(lbl, dtype=torch.long)


# # Compute class weights
# num_classes5 = len(cls2id5)

# class_counts5 = [
#     cnt5.get(i,1)
#     for i in range(num_classes5)
# ]

# total5 = sum(class_counts5)

# weights5 = torch.tensor(
#     [total5 / (num_classes5 * c) for c in class_counts5],
#     dtype=torch.float32
# ).to(DEVICE)


# dl_tr5 = DataLoader(
#     BehaviorDS(tr5, tf5_train),
#     32,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True
# )

# dl_val5 = DataLoader(
#     BehaviorDS(val5, tf5_val),
#     32,
#     shuffle=False,
#     num_workers=2
# )

  Using 10000 images from CBVD-5
  Using 10000 images from BeefBehavior
  CBVD-5 images    : 0
  Beef behavior    : 0
  Total items      : 0
  ⚠️  No labeled folders found. Generating synthetic data.
  feeding     :   202 images
  drinking    :   187 images
  standing    :   220 images
  resting     :   185 images
  walking     :   206 images


In [65]:
# # ── CELL 15 ── MobileNetV3-Small model ────────────────────────────────────

# CHECKPOINT5 = WORK5 / 'behavior_checkpoint.pth'
# # 
# class BehaviorNet(nn.Module):
#     """
#     MobileNetV3-Small: 2.5M params, ~5MB export, runs in <10ms on mobile.
#     """

#     def __init__(self):
#         super().__init__()

#         bb = tvm.mobilenet_v3_small(weights='IMAGENET1K_V1')

#         in_f = bb.classifier[-1].in_features

#         bb.classifier[-1] = nn.Sequential(
#             nn.Dropout(0.4),
#             nn.Linear(in_f, 5)
#         )

#         self.net = bb

#     def forward(self, x):
#         return self.net(x)


# model5 = BehaviorNet().to(DEVICE)

# crit5 = nn.CrossEntropyLoss(weight=weights5)

# opt5 = torch.optim.AdamW(
#     model5.parameters(),
#     lr=1e-3,
#     weight_decay=1e-4
# )

# sch5 = torch.optim.lr_scheduler.CosineAnnealingLR(
#     opt5,
#     T_max=40
# )

# best_f1_5  = 0.0
# start_epoch = 1


# # ── Resume training if checkpoint exists ──────────────────────────────────
# if CHECKPOINT5.exists():

#     print('  Loading previous checkpoint...')

#     ckpt = torch.load(CHECKPOINT5, map_location=DEVICE)

#     model5.load_state_dict(ckpt['model'])
#     opt5.load_state_dict(ckpt['optimizer'])
#     sch5.load_state_dict(ckpt['scheduler'])

#     best_f1_5  = ckpt['best_f1']
#     start_epoch = ckpt['epoch'] + 1

#     print(f'  Resuming from epoch {start_epoch}')


# # ── Training loop ─────────────────────────────────────────────────────────
# for epoch in range(start_epoch, 41):

#     model5.train()

#     running_loss = 0.0

#     for imgs, labs in dl_tr5:

#         imgs = imgs.to(DEVICE)
#         labs = labs.to(DEVICE)

#         opt5.zero_grad()

#         out  = model5(imgs)

#         loss = crit5(out, labs)

#         loss.backward()

#         opt5.step()

#         running_loss += loss.item()

#     sch5.step()


#     # ── Validation ────────────────────────────────────────────────────────
#     if epoch % 5 == 0 or epoch == 40:

#         model5.eval()

#         p5, l5 = [], []

#         with torch.no_grad():

#             for imgs, labs in dl_val5:

#                 preds = model5(imgs.to(DEVICE)).argmax(1).cpu().tolist()

#                 p5 += preds
#                 l5 += labs.tolist()

#         f15 = f1_score(
#             l5,
#             p5,
#             average='weighted',
#             zero_division=0
#         )

#         print(
#             f'  Epoch {epoch:03d} | '
#             f'Loss: {running_loss/len(dl_tr5):.4f} | '
#             f'F1: {f15:.4f}'
#         )

#         # Save best model
#         if f15 > best_f1_5:

#             best_f1_5 = f15

#             torch.save(
#                 model5.state_dict(),
#                 WORK5 / 'best_behavior.pth'
#             )

#             print('  ✅ Best model updated')


#     # ── Save checkpoint every epoch ──────────────────────────────────────
#     torch.save({
#         'epoch'     : epoch,
#         'model'     : model5.state_dict(),
#         'optimizer' : opt5.state_dict(),
#         'scheduler' : sch5.state_dict(),
#         'best_f1'   : best_f1_5,
#     }, CHECKPOINT5)

#     print(f'  💾 Checkpoint saved: epoch {epoch}')


# print(f'\n  Best F1: {best_f1_5:.4f}')


# # ── Export final model ────────────────────────────────────────────────────
# shutil.copy(
#     WORK5 / 'best_behavior.pth',
#     OUT_DIR / 'behavior_classifier.pth'
# )

# json.dump({
#     'function'      : 'feeding_time_behavior_classification',
#     'model'         : 'MobileNetV3-Small',
#     'datasets_used' : [
#         'cbvd-5cow-behavior-video-dataset',
#         'beef-cattle-behavior-data-set'
#     ],
#     'input_size'    : 224,
#     'input_dtype'   : 'float32',
#     'output_size'   : 5,
#     'classes'       : list(cls2id5.keys()),
#     'normalize_mean': [0.485,0.456,0.406],
#     'normalize_std' : [0.229,0.224,0.225],
#     'best_f1'       : round(best_f1_5,4),
#     'flutter_model' : 'assets/models/behavior_classifier.pth',
# }, open(OUT_DIR / 'behavior_meta.json', 'w'), indent=2)

# print('✅ behavior_classifier.pth saved')


# del model5

# gc.collect()

# torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 104MB/s]


  Epoch 005 | F1: 0.1294  (target > 0.85)
  Epoch 010 | F1: 0.0416  (target > 0.85)
  Epoch 015 | F1: 0.1566  (target > 0.85)
  Epoch 020 | F1: 0.1174  (target > 0.85)
  Epoch 025 | F1: 0.0848  (target > 0.85)
  Epoch 030 | F1: 0.1114  (target > 0.85)
  Epoch 035 | F1: 0.1553  (target > 0.85)
  Epoch 040 | F1: 0.1825  (target > 0.85)
  Best F1: 0.1825
✅ behavior_classifier.pth saved
